# L3: Automation

- You will use [Kubeflow Pipelines](https://www.kubeflow.org/docs/components/pipelines/v2/) to orchestrat and automate a workflow. 
- Kubeflow Pipelines is an open source framework. It's like a construction kit for building machine learning pipelines, making it easy to orchestrate and automate complex tasks. 

In [ ]:
# from kfp import dsl
# from kfp import compiler

# # Ignore FutureWarnings in kfp
# import warnings
# warnings.filterwarnings("ignore", 
#                         category=FutureWarning, 
#                         module='kfp.*')

<!-- ## Kubeflow Pipelines

- Kubeflow pipelines consist of two key concepts: Components and pipelines.
- Pipeline components are like self-contained sets of code that perform various steps in your ML workflow, such as, the first step could be preprocessing data, and second step could betraining a model.

### Simple Pipeline Example 

##### Build the pipeline -->

In [101]:
### these are the same 
### jsonl files from the previous lab

### time stamps have been removed so that 
### the files are consistent for all learners
TRAINING_DATA_URI = "./tune_data_stack_overflow_gemini_python_qa-pilot-500.jsonl"
EVALUATION_DATA_URI = "./tune_eval_data_stack_overflow_gemini_python_qa-pilot-100.jsonl"


<!-- - Provide the model with a version.
- Versioning model allows for:
  - Reproducibility: Reproduce your results and ensure your models perform as expected.
  - Auditing: Track changes to your models.
  - Rollbacks: Roll back to a previous version of your model. -->

In [102]:
# The original course template targeted the retired PaLM/Bison tuning flow.
# Current Vertex AI tuning should use the Gemini supervised fine-tuning API instead.
template_path = None

In [103]:
import datetime

In [104]:
date = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [105]:
MODEL_NAME = f"deep-learning-ai-model-01-{date}"

- This example keeps two tuning parameters for demonstration:
  - `TRAINING_STEPS`: Number of training steps to use when tuning the model. For extractive QA you can set it from 100-500. 
  - `EVALUATION_INTERVAL`: The interval determines how frequently a trained model is evaluated against the created *evaluation set* to assess its performance and identify issues. Default will be 20, which means after every 20 training steps, the model is evaluated on the evaluation dataset.

In [106]:
TRAINING_STEPS = 200
EVALUATION_INTERVAL = 20

- Load the Project ID and credentials

In [107]:
from utils import authenticate
credentials, PROJECT_ID = authenticate() 

In [108]:
REGION = "us-central1"

- Define the arguments, the input that goes into the pipeline.

In [109]:
pipeline_arguments = {
    "model_display_name": MODEL_NAME,
    "location": REGION,
    "large_model_reference": "gemini-2.5-flash-lite",
    "project": PROJECT_ID,
    "train_steps": TRAINING_STEPS,
    "dataset_uri": TRAINING_DATA_URI,
    "evaluation_interval": EVALUATION_INTERVAL,
    "evaluation_data_uri": EVALUATION_DATA_URI,
}

In [110]:
pipeline_root = "gs://first-llmops-demo-bucket-2026/pipeline-root"
print("Set this to a real gs:// bucket path before running Gemini supervised tuning.")

Set this to a real gs:// bucket path before running Gemini supervised tuning.


In [111]:
import vertexai
from google.cloud import storage
from vertexai.tuning import sft

vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)

if not pipeline_root.startswith("gs://"):
    raise ValueError("Set pipeline_root to a real gs:// bucket path before submitting tuning.")

bucket_and_prefix = pipeline_root.removeprefix("gs://")
bucket_name, _, prefix = bucket_and_prefix.partition("/")
prefix = prefix.strip("/")

storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)
bucket = storage_client.bucket(bucket_name)

def upload_to_gcs(local_path, destination_name):
    blob_name = f"{prefix}/{destination_name}" if prefix else destination_name
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local_path)
    return f"gs://{bucket_name}/{blob_name}"

train_dataset_gcs_uri = upload_to_gcs(
    TRAINING_DATA_URI,
    f"training/{MODEL_NAME}.jsonl",
)
validation_dataset_gcs_uri = upload_to_gcs(
    EVALUATION_DATA_URI,
    f"validation/{MODEL_NAME}.jsonl",
)

print(train_dataset_gcs_uri)
print(validation_dataset_gcs_uri)

gs://first-llmops-demo-bucket-2026/pipeline-root/training/deep-learning-ai-model-01-2026-04-19_01-44-54.jsonl
gs://first-llmops-demo-bucket-2026/pipeline-root/validation/deep-learning-ai-model-01-2026-04-19_01-44-54.jsonl


In [112]:
tuning_job = sft.train(
    source_model="gemini-2.5-flash-lite",
    train_dataset=train_dataset_gcs_uri,
    validation_dataset=validation_dataset_gcs_uri,
    tuned_model_display_name=MODEL_NAME,
)

print(tuning_job)

Creating SupervisedTuningJob
SupervisedTuningJob created. Resource name: projects/838831985868/locations/us-central1/tuningJobs/6819183055475834880
To use this SupervisedTuningJob in another session:
tuning_job = sft.SupervisedTuningJob('projects/838831985868/locations/us-central1/tuningJobs/6819183055475834880')
View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/us-central1/tuning/tuningJob/6819183055475834880?project=838831985868


resource name: projects/838831985868/locations/us-central1/tuningJobs/6819183055475834880


In [113]:
tuning_job.refresh()
print(tuning_job)
print(f"has_ended={tuning_job.has_ended}")

resource name: projects/838831985868/locations/us-central1/tuningJobs/6819183055475834880
has_ended=False


In [114]:
# print(PROJECT_ID)
# print(pipeline_arguments["project"])


In [115]:
tuning_job.refresh()
print(tuning_job)

resource name: projects/838831985868/locations/us-central1/tuningJobs/6819183055475834880


In [ ]:
print("has_ended =", tuning_job.has_ended)
print("resource_name =", tuning_job.resource_name)
print("state =", tuning_job.state if hasattr(tuning_job, "state") else "no state field")
print("error =", tuning_job.error if hasattr(tuning_job, "error") else "no error field")
print("tuned_model_name =", getattr(tuning_job, "tuned_model_name", "not available yet"))
print("tuned_model_endpoint_name =", getattr(tuning_job, "tuned_model_endpoint_name", "not available yet"))


In [ ]:
print(tuning_job._gca_resource)

# Copy these two values into L3_predictions_prompts_safety once the job finishes:
# - tuned_model_name
# - tuned_model_endpoint_name
